# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshalKushwaha0027/FlyRankAI/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The Playbook Queue:**
This queue translates raw model probabilities into actionable recommendations for the content team.
*   **Urgent Rewrite:** Probability > 70% AND Impressions > 1,000. These are high-traffic pages actively losing ground. Action: Immediate content refresh and metadata optimization.
*   **Monitor:** Probability 40-70%. These pages show early warning signs but haven't crashed yet. Action: Add to a 30-day watchlist before investing editorial hours.
*   **Safe:** Probability < 40%. Expected normal behavior. Action: Do nothing.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Connect to the warehouse and create a sample test_df
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Pulling a sample of pages and assigning a mock prediction probability
query = f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as impressions_90d,
        RANDOM() as pred_prob
    FROM read_parquet('{table_path}')
    WHERE month = '2026-03'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 100
    LIMIT 1000
"""
test_df = con.sql(query).df()

# 2. Apply the action and reason code logic
def assign_action(row):
    if row['pred_prob'] > 0.70 and row['impressions_90d'] > 1000:
        return pd.Series(['Urgent Rewrite', 'high_risk_high_volume'])
    elif row['pred_prob'] > 0.40:
        return pd.Series(['Monitor', 'borderline_risk'])
    else:
        return pd.Series(['Safe', 'normal_behavior'])

test_df[['action', 'reason_code']] = test_df.apply(assign_action, axis=1)

# 3. Sort by risk and traffic to build the queue
playbook_queue = test_df.sort_values(by=['pred_prob', 'impressions_90d'], ascending=[False, False])
display(playbook_queue[['content_hash_id', 'pred_prob', 'impressions_90d', 'action', 'reason_code']].head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,pred_prob,impressions_90d,action,reason_code
321,content_2f7a336c4d3fae2d,0.997372,10240.0,Urgent Rewrite,high_risk_high_volume
380,content_cca2a04092ad93eb,0.996497,467.0,Monitor,borderline_risk
979,content_0cf8001343982c36,0.996024,1608.0,Urgent Rewrite,high_risk_high_volume
739,content_46b04cc3ddffcb45,0.994911,178.0,Monitor,borderline_risk
236,content_fce32601b86f7bc9,0.994152,1609.0,Urgent Rewrite,high_risk_high_volume
90,content_d6e6256a18d0a053,0.993600,534.0,Monitor,borderline_risk
71,content_35b28cf3598c1c40,0.991633,1774.0,Urgent Rewrite,high_risk_high_volume
497,content_7857688a198f529a,0.991162,1112.0,Urgent Rewrite,high_risk_high_volume
304,content_6138860d39ca5233,0.990784,5195.0,Urgent Rewrite,high_risk_high_volume
609,content_dc1c28ea3dde8f3f,0.989983,231.0,Monitor,borderline_risk


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:**
This model is a **directional decision-support tool** for SEO and content teams. It is designed to flag *risk* and prioritize workflow, not to autonomously edit live website code.

**Limits:**
This model operates entirely on historical on-page metrics. It is blind to off-page variables. It will not know if a traffic drop was caused by a competitor doubling their ad spend, a lost high-value backlink, or a seasonal shift in user behavior.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No computation needed here
print("✅ Intended use and limits successfully defined.")


✅ Intended use and limits successfully defined.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**The No-Go List:**
Automated flagging should never bypass human editorial guidelines. The following page archetypes must be excluded from this workflow, regardless of their risk score:
1. **Legal & Compliance:** Privacy policies, terms of service, and cookie disclosures.
2. **Navigational/Utility:** Contact us pages, login portals, and password resets.
3. **Recent Publications:** Any content published or significantly updated within the last 30 days (they need time to rank).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("✅ No-Go list established for human review guardrails.")


✅ No-Go list established for human review guardrails.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain Triggers:**
This model's baseline assumptions will eventually drift. We must pause recommendations and trigger a retraining cycle if:
1. **Google Core Update:** A confirmed, major algorithm rollout changes fundamental ranking weights.
2. **Site Migration:** The domain undergoes a structural overhaul (URL changes, massive design shifts).
3. **Data Drift:** The global median CTR for the site drops by more than 20% week-over-week, indicating a tracking error or SERP layout change.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("✅ Retrain and monitoring triggers defined.")


✅ Retrain and monitoring triggers defined.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Exporting the Queue:**
Exporting the ranked playbook queue to the `work/outputs/` directory. This file will be referenced by the final capstone research paper.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# Ensure the output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Export the queue to CSV (excluding index)
output_path = 'work/outputs/action_playbook_queue.csv'
playbook_queue.to_csv(output_path, index=False)

print(f"✅ Playbook queue successfully exported to: {output_path}")


✅ Playbook queue successfully exported to: work/outputs/action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.